In [72]:
from pathlib import Path
import pandas as pd
import numpy as np
COURSE = Path('C:/Users/ingra/Documents/Ironhack') #Course path
#sales = pd.read_csv(COURSE/'labs/Mini-Project-Deliverables/sales.csv') #Read the data with course path
sales = pd.read_csv('./sales.csv') #Read the data
real_data = pd.read_csv('./REAL_DATA - REAL_DATA.csv') #Read the data


In [73]:
sales.head() #Show the first 5 rows of the data

,Unnamed: 0,store_ID,day_of_week,date,nb_customers_on_day,open,promotion,state_holiday,school_holiday,sales
0,425390,366,4,2013-04-18,517,1,0,0,0,4422
1,291687,394,6,2015-04-11,694,1,0,0,0,8297
2,411278,807,4,2013-08-29,970,1,1,0,0,9729
3,664714,802,2,2013-05-28,473,1,1,0,0,6513
4,540835,726,4,2013-10-10,1068,1,1,0,0,10882


In [74]:
real_data.head() #Show the first 5 rows of the data

,index,store_ID,day_of_week,date,nb_customers_on_day,open,promotion,state_holiday,school_holiday
0,272371,415,7,01/03/2015,0,0,0,0,0
1,558468,27,7,29/12/2013,0,0,0,0,0
2,76950,404,3,19/03/2014,657,1,1,0,0
3,77556,683,2,29/01/2013,862,1,0,0,0
4,456344,920,3,19/03/2014,591,1,1,0,0


In [75]:
df = sales.copy() #Create a copy of the original DataFrame to work with
df = df.drop(columns=['Unnamed: 0']) #Drop the 'Unnamed: 0' column
df['state_holiday'] = df['state_holiday'].astype(str) #Convert everything to string
df.info() #Display information about the DataFrame, including data types and non-null counts
# for col in df2.columns: 
#     print(col, df2[col].value_counts().to_dict())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 640840 entries, 0 to 640839
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   store_ID             640840 non-null  int64 
 1   day_of_week          640840 non-null  int64 
 2   date                 640840 non-null  object
 3   nb_customers_on_day  640840 non-null  int64 
 4   open                 640840 non-null  int64 
 5   promotion            640840 non-null  int64 
 6   state_holiday        640840 non-null  object
 7   school_holiday       640840 non-null  int64 
 8   sales                640840 non-null  int64 
dtypes: int64(7), object(2)
memory usage: 44.0+ MB


In [76]:
missing_values = pd.isnull(df)
missing_counts = missing_values.sum()
columns_with_missing = missing_counts[missing_counts > 0].count()
all_columns_missing = missing_counts.all()
total_missing_values = missing_counts.sum()
print("Missing Values in Each Column:\n", missing_counts)
print("\nNumber of Columns with Missing Values:", columns_with_missing)
print("All Columns Have Missing Values:", all_columns_missing)
print("\nTotal Missing Values in the DataFrame:", total_missing_values)

Missing Values in Each Column:
 store_ID               0
day_of_week            0
date                   0
nb_customers_on_day    0
open                   0
promotion              0
state_holiday          0
school_holiday         0
sales                  0
dtype: int64

Number of Columns with Missing Values: 0
All Columns Have Missing Values: False

Total Missing Values in the DataFrame: 0


In [78]:
from sklearn import linear_model #Importo el módulo linear_model de la biblioteca sklearn, que contiene clases y funciones para crear modelos de regresión lineal.
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score 
from sklearn.model_selection import train_test_split #Importo la función train_test_split del módulo model_selection de sklearn para dividir los datos en conjuntos de entrenamiento y prueba.

# Crear nuevas columnas para año, mes y día
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day

#state_holiday {'0': 621160, 'a': 12842, 'b': 4214, 'c': 2624}

state_holiday_mapping = {'0': 0, 'a': 1, 'b': 2, 'c': 3} 
df['state_holiday'] = df['state_holiday'].map(state_holiday_mapping)


# 1. Definir variables X (features=Características) y y (lo que quiero predecir)
features = ['store_ID', 'day_of_week', 'nb_customers_on_day', 'open', 'promotion', 'school_holiday', 'year', 'month', 'day']
x = df[features]
y = df['sales']

# 2. Dividir los datos train vs test (80% train, 20% test)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# 3. Entrenar el modelo
model = linear_model.LinearRegression() #llamo a la clase LinearRegression del módulo linear_model de sklearn para crear un objeto regr que representa un modelo de regresión lineal.
model.fit(x_train, y_train) #Entreno el modelo de regresión lineal utilizando los datos de entrenamiento (train_x y train_y).


# 4. Imprimo los coeficientes y la intersección del modelo de regresión lineal entrenado.
print ('Coefficients: ', model.coef_) #Los coeficientes representan la pendiente de la línea de regresión
print ('Intercept: ',model.intercept_) #la intersección representa el valor de salida cuando la entrada es cero.

Coefficients:  [-1.44423660e-01 -5.69342874e+01  6.06864308e+00  1.67957454e+03
  1.27999834e+03 -4.26840143e+00  1.43481400e+02  3.48140308e+01
  1.42582921e+00]
Intercept:  -288812.7585391804


AHORA PROCEDEMOS A HALLAR EL R2

In [84]:
from sklearn.metrics import r2_score
y_pred = model.predict(x_test) #Hago predicciones utilizando el modelo entrenado y los datos de prueba (x_test).
y_pred = np.maximum(y_pred, 0) #Aseguro que las predicciones no sean negativas, ya que las ventas no pueden ser negativas.
r2_score_v1 = 0.7984 #features = ['nb_customers_on_day']
r2_score_v2 = 0.8501 #features = ['store_ID', 'day_of_week', 'open', 'promotion', 'school_holiday', 'nb_customers_on_day']
r2_score_v3 = 0.8514 #features = ['store_ID', 'day_of_week', 'nb_customers_on_day', 'open', 'promotion', 'school_holiday', 'year', 'month', 'day']
r2_score_v4 = r2_score(y_test, y_pred) #ventas no negativas
print(f"R2 SCORE MODELO VERSION 1: {r2_score_v1:.4f}")
print(f"R2 SCORE MODELO VERSION 2: {r2_score_v2:.4f}")
print(f"R2 SCORE MODELO VERSION 3: {r2_score_v3:.4f}")
print(f"R2 SCORE MODELO VERSION 4: {r2_score_v4:.4f}")

R2 SCORE MODELO VERSION 1: 0.7984
R2 SCORE MODELO VERSION 2: 0.8501
R2 SCORE MODELO VERSION 3: 0.8514
R2 SCORE MODELO VERSION 4: 0.8517


### PRONOSTICAMOS EN EL ARCHIVO REAL DATA USANDO NUESTRO MODELO ENTRENADO

In [85]:
df2 = real_data.copy() #Create a copy of the original DataFrame to work with
df2['state_holiday'] = df2['state_holiday'].astype(str) #Convert everything to string
df2.info() #Display information about the DataFrame, including data types and non-null counts

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71205 entries, 0 to 71204
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   index                71205 non-null  int64  
 1   store_ID             71205 non-null  int64  
 2   day_of_week          71205 non-null  int64  
 3   date                 71205 non-null  object 
 4   nb_customers_on_day  71205 non-null  int64  
 5   open                 71205 non-null  int64  
 6   promotion            71205 non-null  int64  
 7   state_holiday        71205 non-null  object 
 8   school_holiday       71205 non-null  int64  
 9   Prediction           71205 non-null  float64
dtypes: float64(1), int64(7), object(2)
memory usage: 5.4+ MB


In [86]:
missing_values = pd.isnull(df2)
missing_counts = missing_values.sum()
columns_with_missing = missing_counts[missing_counts > 0].count()
all_columns_missing = missing_counts.all()
total_missing_values = missing_counts.sum()
print("Missing Values in Each Column:\n", missing_counts)
print("\nNumber of Columns with Missing Values:", columns_with_missing)
print("All Columns Have Missing Values:", all_columns_missing)
print("\nTotal Missing Values in the DataFrame:", total_missing_values)

Missing Values in Each Column:
 index                  0
store_ID               0
day_of_week            0
date                   0
nb_customers_on_day    0
open                   0
promotion              0
state_holiday          0
school_holiday         0
Prediction             0
dtype: int64

Number of Columns with Missing Values: 0
All Columns Have Missing Values: False

Total Missing Values in the DataFrame: 0


In [87]:
# Crear nuevas columnas para año, mes y día
df2['date'] = pd.to_datetime(df2['date'], dayfirst=True)
df2['year'] = df2['date'].dt.year
df2['month'] = df2['date'].dt.month
df2['day'] = df2['date'].dt.day

#state_holiday {'0': 621160, 'a': 12842, 'b': 4214, 'c': 2624}

state_holiday_mapping = {'0': 0, 'a': 1, 'b': 2, 'c': 3} 
df2['state_holiday'] = df2['state_holiday'].map(state_holiday_mapping)

df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71205 entries, 0 to 71204
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   index                71205 non-null  int64         
 1   store_ID             71205 non-null  int64         
 2   day_of_week          71205 non-null  int64         
 3   date                 71205 non-null  datetime64[ns]
 4   nb_customers_on_day  71205 non-null  int64         
 5   open                 71205 non-null  int64         
 6   promotion            71205 non-null  int64         
 7   state_holiday        71205 non-null  int64         
 8   school_holiday       71205 non-null  int64         
 9   Prediction           71205 non-null  float64       
 10  year                 71205 non-null  int32         
 11  month                71205 non-null  int32         
 12  day                  71205 non-null  int32         
dtypes: datetime64[ns](1), float64(1

In [ ]:
features2 = ['store_ID', 'day_of_week', 'nb_customers_on_day', 'open', 'promotion', 'school_holiday', 'year', 'month', 'day']
x2 = df2[features2]
y_pred_2 = model.predict(x2) #Hago predicciones utilizando el modelo entrenado y los datos de real data.
y_pred_2 = np.maximum(y_pred_2, 0) #Aseguro que las predicciones no sean negativas
real_data['Prediction'] = y_pred_2 # Asigno las predicciones al DataFrame de datos reales.
real_data.head()

,index,store_ID,day_of_week,date,nb_customers_on_day,open,promotion,state_holiday,school_holiday,Prediction
0,272371,415,7,01/03/2015,0,0,0,0,0,0.000000
1,558468,27,7,29/12/2013,0,0,0,0,0,71.976632
2,76950,404,3,19/03/2014,657,1,1,0,0,7007.834273
3,77556,683,2,29/01/2013,862,1,0,0,0,6789.696684
4,456344,920,3,19/03/2014,591,1,1,0,0,6532.781221
